In [1]:
import sklearn
import torch.nn as nn
import torch.optim as optim
import torch
import numpy as np

In [2]:
import sys
print(sys.version)
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.version.cuda)

3.11.3 (main, Sep 17 2024, 22:50:04) [GCC 12.3.0]
2.12.1+cu130
False
0
13.0


# Using ML libraries
Thankfully we don't have to implement the models or metrics from scratch. There are some python libraries that have us covered. In this module, I'll explain their common use cases.

## scikit-learn
Scikit-learn provides datasets, metrics, and some simpler models available to train. If you are building a random forest, running a linear regression, or splitting your data into training and testing sets, this is your go-to tool.

In [7]:
# import model, train_test_split, wine dataset, and metrics through sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_wine
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer

# You can check which models are available
from sklearn.utils.discovery import all_estimators

estimators = all_estimators()

for name, class_ in estimators:
    print(name)

ARDRegression
AdaBoostClassifier
AdaBoostRegressor
AdditiveChi2Sampler
AffinityPropagation
AgglomerativeClustering
BaggingClassifier
BaggingRegressor
BayesianGaussianMixture
BayesianRidge
BernoulliNB
BernoulliRBM
Binarizer
Birch
BisectingKMeans
CCA
CalibratedClassifierCV
CategoricalNB
ClassicalMDS
ClassifierChain
ColumnTransformer
ComplementNB
CountVectorizer
DBSCAN
DecisionTreeClassifier
DecisionTreeRegressor
DictVectorizer
DictionaryLearning
DummyClassifier
DummyRegressor
ElasticNet
ElasticNetCV
EllipticEnvelope
EmpiricalCovariance
ExtraTreeClassifier
ExtraTreeRegressor
ExtraTreesClassifier
ExtraTreesRegressor
FactorAnalysis
FastICA
FeatureAgglomeration
FeatureHasher
FeatureUnion
FixedThresholdClassifier
FrozenEstimator
FunctionTransformer
GammaRegressor
GaussianMixture
GaussianNB
GaussianProcessClassifier
GaussianProcessRegressor
GaussianRandomProjection
GenericUnivariateSelect
GradientBoostingClassifier
GradientBoostingRegressor
GraphicalLasso
GraphicalLassoCV
GridSearchCV
HDBSCAN


In [8]:
# loading data
wine = load_wine()
X = wine.data
y = wine.target
feature_names = wine.feature_names
print("Feature names: ", feature_names)

# split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

# call and fit random forest
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# Make predictions on the unseen test data
y_pred = model.predict(X_test)

# Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

Feature names:  ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']
Test Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54



## Pytorch

When you need to apply deep learning models, Pytorch and Tensorflow are the optimal choices. We will be using PyTorch since it is the standard for research. 

Pytorch treats everything as a Tensor (a multi-dimensional matrix) and allows you to run heavy mathematical operations on a GPU to speed things up. It uses Dataset and DataLoader to manage data. The DataLoader automatically splits your data into small management-sized chunks called batches and shuffles them. Typically in Deep Learning you would not train over the entire dataset in one sweep. You would train in batches which would update the model's internal weights. This is done to prevent the system from running out of hardware memory and balances optimization speed. 

Additionally, it also gives important functions for us, such as gradient descent, loss functions, activation functions, etc to make our lives way easier.

Here we will replicate the NN we made in a previous notebook and use the wine dataset. You'll notice how much simpler the code is now. I will also use a multiclass version. So instead of the sigmoid activation function we can have cross entropy loss deal with the classes internally.

In [12]:
class NN_Adam(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, learning_rate=0.01):
        super().__init__()
        
        # Define the network architecture
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),  # w1, b1 inside here
            nn.ReLU(),
            nn.Linear(hidden_size, num_classes)           # w2, b2 inside here
        )
        
        # weight initialization
        with torch.no_grad():
            nn.init.normal_(self.network[0].weight, mean=0.0, std=0.01)
            nn.init.zeros_(self.network[0].bias)
            nn.init.normal_(self.network[2].weight, mean=0.0, std=0.01)
            nn.init.zeros_(self.network[2].bias)

        # PyTorch built-in Adam Optimizer and Loss Function
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)
        self.criterion = nn.CrossEntropyLoss()
        
        self.loss_history = []

    def forward(self, X):
        # Data will automatically pass through
        return self.network(X)

    def train_model(self, X, y, epochs=100):
        self.train()   # Set model to training mode

        y = y.view(-1)
        for e in range(epochs):
            # Forward Pass
            y_hat = self.forward(X)
            loss = self.criterion(y_hat, y)
            self.loss_history.append(loss.item())

            # Backward Pass
            self.optimizer.zero_grad()   # Clear previous gradients
            loss.backward()   # Calculate new gradients
            
            # Optimization Step
            self.optimizer.step()   # Update weights (w1, b1, w2, b2)

            if e % 10 == 0:
                print(f"Epoch {e:4d} | Loss: {loss.item():.4f}")
                
        return self.loss_history

    def predict(self, X):
        self.eval() # Set model to evaluation mode
        with torch.no_grad():
            logits = self.forward(X)
            return torch.argmax(logits, dim=1) 

In [13]:
# split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Use StandardScaler from SciKit-Learn
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [14]:
# checking labels

print(y_train[:10])
print(np.unique(y_train))

[2 2 1 2 0 1 1 1 2 0]
[0 1 2]


In [15]:
# convert data over to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print(X_train_tensor.shape)
print(y_train_tensor.shape)
print(y_train_tensor.dtype)

torch.Size([142, 13])
torch.Size([142])
torch.int64


In [ ]:
# Initialize and Train the PyTorch Model
input_dim = X_train.shape[1]
hidden_dim = 8

model = NN_Adam(input_size=input_dim, hidden_size=hidden_dim, num_classes = 3, learning_rate=0.01)
losses = model.train_model(X_train_tensor, y_train_tensor, epochs=100)

My kernel keeps crashing whenever I attempt to train the model. Part of the issue is my pytorch is not using CUDA which means it is completely CPU bound. This crashes the kernel because it is overloaded. I'll be real, I am a little too lazy to fix it. See if you can output the loss plots and determine which metrics you think will be useful in evaluating the sucesses and failures of our approach.